# 🎯 Face RTRecord — Full Pipeline on Google Colab

> Complete face attendance system: Record → Extract → Augment → Train → Recognize

**Full port of 01_record.py, 02_extract.py, 02_augment.py, 03_train.py, 04_recognize.py + config.py**

## 📁 Setup Folders in Google Drive
```
MyDrive/face_RTRecord/
├── videos/     ← upload videos here
├── dataset/    ← extracted faces (auto)
├── dataset_aug/← augmented (auto)
├── embeddings/ ← DB (auto)
└── logs/       ← attendance (auto)
```

In [ ]:
# BƯỚC 0: INSTALL & MOUNT
!pip install deepface==0.0.93 tf-keras>=2.15 tensorflow-cpu>=2.15.0 opencv-python albumentations scipy tqdm openpyxl pandas retina-face -q

from google.colab import drive
drive.mount('/content/drive')

from google.colab.patches import cv2_imshow
from google.colab import files
import os, glob, pickle, time, hashlib, numpy as np, cv2, pandas as pd
from tqdm import tqdm
from pathlib import Path
from datetime import datetime
from scipy.spatial.distance import cosine
from deepface import DeepFace
from IPython.display import display, Image
from PIL import Image as PILImage

print('✅ Setup complete!')

In [ ]:
# CONFIG (from config.py)
BASE_DIR = Path('/content/drive/MyDrive/face_RTRecord')
VIDEO_DIR = BASE_DIR / 'videos'
DATASET_DIR = BASE_DIR / 'dataset'
DATASET_AUG_DIR = BASE_DIR / 'dataset_aug'
EMBED_DIR = BASE_DIR / 'embeddings'
LOG_DIR = BASE_DIR / 'logs'

for d in [VIDEO_DIR, DATASET_DIR, DATASET_AUG_DIR, EMBED_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Params
MODEL_NAME = 'ArcFace'
DETECTOR_BACKEND = 'opencv'  # or 'retinaface'
THRESHOLD = 0.40
CAPTURE_EVERY_N_FRAMES = 8
MAX_IMAGES_PER_PERSON = 100
MIN_IMAGES_PER_PERSON = 60
MIN_FACE_SIZE = 80
BLUR_THRESHOLD = 100.0
MIN_BRIGHTNESS = 40
MAX_BRIGHTNESS = 230
DUPLICATE_THRESHOLD = 8

print('📁 Folders ready')

## 📹 BƯỚC 1: Record (01_record.py port)

**Upload video file** (record locally, upload to videos/)

In [ ]:
# Upload video (simulate 01_record)
print('📤 Upload MP4 video to videos/ folder')
uploaded = files.upload()

for fn in uploaded.keys():
  video_path = VIDEO_DIR / fn
  with open(video_path, 'wb') as f:
    f.write(uploaded[fn])
  print(f'💾 Saved: {video_path}')

# List videos
!ls -la /content/drive/MyDrive/face_RTRecord/videos/

## 🔍 BƯỚC 2: Extract Faces from Video (02_extract.py port)

In [ ]:
# 02_extract.py functions
def laplacian_score(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

def brightness_ok(img, lo, hi):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    m = gray.mean()
    return lo <= m <= hi, m

def phash(img):
    small = cv2.resize(img, (32, 32))
    gray  = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY).astype(float)
    dct   = cv2.dct(gray)[:8, :8]
    return (dct > np.median(dct)).flatten()

def hamming(h1, h2):
    return int(np.count_nonzero(h1 != h2))

def is_too_similar(face_img, existing_hashes, threshold):
    h = phash(face_img)
    for prev in existing_hashes[-40:]:
        if hamming(h, prev) <= threshold:
            return True
    return False

def load_opencv_detector():
    cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    return cascade

def detect_faces(frame, detector):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)
    rects = detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))
    return [(x, y, w, h) for (x, y, w, h) in rects]

def extract_faces(name, video_path, max_images=MAX_IMAGES_PER_PERSON):
    save_dir = DATASET_DIR / name
    save_dir.mkdir(exist_ok=True)

    existing = list(save_dir.glob('*.jpg'))
    start_idx = len(existing)

    cap = cv2.VideoCapture(str(video_path))
    detector = load_opencv_detector()
    saved = 0
    phashes = []
    frame_idx = 0

    print(f'Extracting {name} from {video_path}')
    pbar = tqdm(total=max_images, desc=name)

    while saved < max_images:
        ret, frame = cap.read()
        if not ret: break

        frame_idx += 1
        if frame_idx % CAPTURE_EVERY_N_FRAMES != 0: continue

        faces = detect_faces(frame, detector)
        if not faces: continue

        x, y, w, h = sorted(faces, key=lambda f: f[2]*f[3], reverse=True)[0]
        if w < MIN_FACE_SIZE: continue

        pad = int(min(w, h) * 0.15)
        x1, y1 = max(0, x-pad), max(0, y-pad)
        x2, y2 = min(frame.shape[1], x+w+pad), min(frame.shape[0], y+h+pad)
        face = cv2.resize(frame[y1:y2, x1:x2], (224, 224))

        if laplacian_score(face) < BLUR_THRESHOLD: continue
        if not brightness_ok(face, MIN_BRIGHTNESS, MAX_BRIGHTNESS)[0]: continue
        if phashes and is_too_similar(face, phashes, DUPLICATE_THRESHOLD): continue

        path = save_dir / f'{name}_{start_idx + saved:04d}.jpg'
        cv2.imwrite(str(path), face)
        phashes.append(phash(face))
        saved += 1
        pbar.update(1)

    pbar.close()
    cap.release()
    print(f'✅ Saved {saved} faces for {name}')
    return saved

In [ ]:
# RUN Extract (input name and video)
name = input('Person name (e.g. ThanHieu1): ').strip()
videos = list(VIDEO_DIR.glob('*.mp4'))
if videos:
    video_path = videos[-1]
else:
    print('No video found. Upload first.')
    video_path = None

if video_path:
    extract_faces(name, video_path)

# Check dataset
!ls -la /content/drive/MyDrive/face_RTRecord/dataset/

## 🔄 BƯỚC 3: Augment (02_augment.py port)

In [ ]:
# 02_augment.py
import albumentations as A

def augment_images(name):
    input_dir = DATASET_DIR / name
    output_dir = DATASET_AUG_DIR / name
    output_dir.mkdir(exist_ok=True)

    transform = A.Compose([
        A.RandomBrightnessContrast(p=0.5),
        A.GaussNoise(p=0.3),
        A.MotionBlur(p=0.2),
        A.Rotate(limit=20, p=0.5),
        A.HorizontalFlip(p=0.5)
    ])

    imgs = list(input_dir.glob('*.jpg'))
    aug_per = 5

    for img_path in tqdm(imgs, desc=f'Augment {name}'):
        img = cv2.imread(str(img_path))
        name_base, _ = os.path.splitext(img_path.name)
        for i in range(aug_per):
            aug = transform(image=img)['image']
            cv2.imwrite(str(output_dir / f'{name_base}_aug_{i}.jpg'), aug)

# Run augment for person
name = input('Person name to augment: ').strip()
augment_images(name)
print(f'✅ Augmented to {DATASET_AUG_DIR / name}')

## 🧠 BƯỚC 4: Train Embeddings (03_train.py port)

In [ ]:
# 03_train.py
def get_embedding(img_path):
    try:
        result = DeepFace.represent(img_path, model_name=MODEL_NAME, detector_backend=DETECTOR_BACKEND,
                                   enforce_detection=False, align=True)
        if result:
            return np.array(result[0]['embedding'])
    except:
        pass
    return None

def build_db():
    db = {}
    persons = [d.name for d in DATASET_AUG_DIR.iterdir() if d.is_dir()]

    for person in tqdm(persons, desc='Building DB'):
        imgs = glob.glob(str(DATASET_AUG_DIR / person / '*.jpg'))
        embeds = [get_embedding(img) for img in imgs if get_embedding(img) is not None]
        if embeds:
            mean_emb = np.mean(embeds, axis=0)
            norm = np.linalg.norm(mean_emb)
            if norm > 0:
                mean_emb /= norm
            db[person] = {'mean': mean_emb, 'count': len(embeds)}

    embed_path = EMBED_DIR / f'db_{MODEL_NAME.lower()}.pkl'
    with open(embed_path, 'wb') as f:
        pickle.dump(db, f)
    print(f'💾 DB saved: {embed_path} | {len(db)} persons')
    return db

db = build_db()

## 🎯 BƯỚC 5: Recognize (04_recognize.py port + JS Webcam)

In [ ]:
# Load DB
embed_path = list(EMBED_DIR.glob('db_*.pkl'))[0]
with open(embed_path, 'rb') as f:
    db = pickle.load(f)

def recognize(img_path):
    emb = get_embedding(img_path)
    if emb is None:
        return 'NO_FACE', 0.0, 1.0
    norm = np.linalg.norm(emb)
    if norm > 0:
        emb /= norm
    best = 'UNKNOWN'
    best_dist = float('inf')
    for person, data in db.items():
        dist = cosine(emb, data['mean'])
        if dist < best_dist:
            best_dist = dist
            best = person
    conf = (1 - best_dist) * 100 if best_dist <= THRESHOLD else 0
    return best, conf, best_dist

# Test with upload
print('Upload image for test')
uploaded = files.upload()
for fn, data in uploaded.items():
    path = f'/tmp/{fn}'
    with open(path, 'wb') as f:
        f.write(data)
    name, conf, dist = recognize(path)
    print(f'{name} | {conf:.1f}% | dist={dist:.3f}')
    display(Image(path))

In [ ]:
# JS Webcam for live recognize (04_recognize UI)
from IPython.display import HTML, Javascript
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.9):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

// Resize video to fit
      google.colab.output.setIframeHeight(document.querySelector('.output').scrollHeight, true);

// Wait 2s then capture
      await new Promise((resolve) => setTimeout(resolve, 2000));
      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getTracks().forEach(track => track.stop());
      div.remove();
      canvas.toBlob(blob => {
        const reader = new FileReader();
        reader.readAsDataURL(blob);
        reader.onloadend = () => {
          const imgString = reader.result.split(',')[1];
          google.colab.kernel.invokeFunction('notebook.save_image', [imgString], {});
        };
      }, 'image/jpeg', quality);
    }
    takePhoto(%s);
  ''' % quality)
  display(js)

def save_image(img_b64):
  img_data = b64decode(img_b64)
  with open('/tmp/photo.jpg', 'wb') as f:
    f.write(img_data)
  name, conf, dist = recognize('/tmp/photo.jpg')
  print(f'Live result: {name} ({conf:.1f}%)')
  cv2_imshow(cv2.imread('/tmp/photo.jpg'))

take_photo()

## 📋 Attendance Log & Report

In [ ]:
# Log attendance
def log_attendance(person, conf):
    today = datetime.now().strftime('%Y-%m-%d')
    log_file = LOG_DIR / f'attendance_{today}.csv'
    new = pd.DataFrame({'name': [person], 'time': [datetime.now()], 'conf': [conf]})
    if log_file.exists():
        df = pd.read_csv(log_file)
        df = pd.concat([df, new])
    else:
        df = new
    df.to_csv(log_file, index=False)
    print(f'Logged: {person} {conf}%')

# View report
!ls -la /content/drive/MyDrive/face_RTRecord/logs/
log_files = list(LOG_DIR.glob('*.csv'))
if log_files:
    df = pd.read_csv(log_files[-1])
    display(df)